In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("Online Retail.xlsx")
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

In [2]:
memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"\nMemory usage: {memory_mb:.2f} MB")

missing_pct = (df.isnull().sum() / len(df)) * 100
print(f"\nMissing value percentages (%):\n{missing_pct.round(2)}")

# 3. Number of unique values per column
print(f"\nUnique values per column:\n{df.nunique()}")

# 4. Basic statistics for numeric columns (min, max, mean, median, std)
# describe() defaults to numeric columns. We transpose (.T) for readability 
# and select the specific columns required.
stats = df.describe().T[['min', 'max', 'mean', '50%', 'std']]
stats.rename(columns={'50%': 'median'}, inplace=True)
print(f"\nBasic statistics for numeric columns:\n{stats.round(2)}")


Memory usage: 104.99 MB

Missing value percentages (%):
InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64

Unique values per column:
InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
dtype: int64

Basic statistics for numeric columns:
                             min                  max  \
Quantity                -80995.0              80995.0   
InvoiceDate  2010-12-01 08:26:00  2011-12-09 12:50:00   
UnitPrice              -11062.06              38970.0   
CustomerID               12346.0              18287.0   

                                   mean               median          std  
Quantity                        9.55225                  3.0   218.081158  
InvoiceDate  2011-07-04 13:34:57.156386  2011-07-19 17:17:00          NaN  

In [3]:
missing_rows_mask = df.isnull().any(axis=1)

df_missing_only = df[missing_rows_mask]

df_missing_only

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
541536,581498,85099B,JUMBO BAG RED RETROSPOT,5,2011-12-09 10:26:00,4.13,NaN,United Kingdom
541537,581498,85099C,JUMBO BAG BAROQUE BLACK WHITE,4,2011-12-09 10:26:00,4.13,NaN,United Kingdom
541538,581498,85150,LADIES & GENTLEMEN METAL SIGN,1,2011-12-09 10:26:00,4.96,NaN,United Kingdom
541539,581498,85174,S/4 CACTI CANDLES,1,2011-12-09 10:26:00,10.79,NaN,United Kingdom


In [4]:
import pandas as pd

# Assuming you already loaded df
# df = pd.read_excel("Online Retail.xlsx")

# Create indicator columns
df['Missing_CustomerID'] = df['CustomerID'].isnull()
df['Missing_Description'] = df['Description'].isnull()

In [5]:
print("\n--- Investigating Missing Descriptions ---")

# 1. How does UnitPrice look when Description is missing vs present?
desc_price_stats = df.groupby('Missing_Description')['UnitPrice'].describe()
print("\nUnitPrice stats based on Missing Description:")
print(desc_price_stats[['count', 'mean', 'min', 'max']])

# 2. Are CustomerIDs also missing when Description is missing?
# A crosstab shows the frequency of combinations
desc_cust_cross = pd.crosstab(df['Missing_Description'], df['Missing_CustomerID'])
print("\nCrosstab: Missing Description vs Missing CustomerID:")
print(desc_cust_cross)


--- Investigating Missing Descriptions ---

UnitPrice stats based on Missing Description:
                        count      mean       min      max
Missing_Description                                       
False                540455.0  4.623519 -11062.06  38970.0
True                   1454.0  0.000000      0.00      0.0

Crosstab: Missing Description vs Missing CustomerID:
Missing_CustomerID    False   True 
Missing_Description                
False                406829  133626
True                      0    1454


In [6]:
print("\n--- Investigating Missing CustomerIDs ---")

# 1. Compare purchasing behavior (Quantity and UnitPrice)
cust_numeric_stats = df.groupby('Missing_CustomerID')[['Quantity', 'UnitPrice']].describe()

print("\nQuantity stats based on Missing CustomerID:")
# .xs is just a pandas trick to grab a specific sub-column easily from the grouped describe()
print(cust_numeric_stats.xs('Quantity', axis=1, level=0)[['mean', 'min', 'max']])

print("\nUnitPrice stats based on Missing CustomerID:")
print(cust_numeric_stats.xs('UnitPrice', axis=1, level=0)[['mean', 'min', 'max']])

# 2. Compare the Country distribution
print("\nTop 5 Countries when CustomerID is Missing (Percentages):")
missing_cust_countries = df[df['Missing_CustomerID']]['Country'].value_counts(normalize=True).head() * 100
print(missing_cust_countries.round(2))

print("\nTop 5 Countries when CustomerID is Present (Percentages):")
present_cust_countries = df[~df['Missing_CustomerID']]['Country'].value_counts(normalize=True).head() * 100
print(present_cust_countries.round(2))


--- Investigating Missing CustomerIDs ---

Quantity stats based on Missing CustomerID:
                         mean      min      max
Missing_CustomerID                             
False               12.061303 -80995.0  80995.0
True                 1.995573  -9600.0   5568.0

UnitPrice stats based on Missing CustomerID:
                        mean       min       max
Missing_CustomerID                              
False               3.460471      0.00  38970.00
True                8.076577 -11062.06  17836.46

Top 5 Countries when CustomerID is Missing (Percentages):
Country
United Kingdom    98.90
EIRE               0.53
Hong Kong          0.21
Unspecified        0.15
Switzerland        0.09
Name: proportion, dtype: float64

Top 5 Countries when CustomerID is Present (Percentages):
Country
United Kingdom    88.95
Germany            2.33
France             2.09
EIRE               1.84
Spain              0.62
Name: proportion, dtype: float64


In [7]:
# 1. Convert InvoiceNo to string so we can check the first letter safely
df['InvoiceNo_Str'] = df['InvoiceNo'].astype(str)

# 2. Create a flag for cancellations (starts with 'C')
df['Is_Cancelled'] = df['InvoiceNo_Str'].str.startswith('C')

# 3. Filter only the rows where Quantity is negative
negative_qty_df = df[df['Quantity'] < 0]

# 4. Create a crosstab to see the overlap
print("\n--- Negative Quantities: Missing CustomerID vs. Cancellations ---")
cancel_check = pd.crosstab(
    negative_qty_df['Missing_CustomerID'], 
    negative_qty_df['Is_Cancelled'],
    rownames=['Missing CustomerID (True/False)'],
    colnames=['Starts with "C" (True/False)']
)
print(cancel_check)

# 5. Let's look directly at the rows where UnitPrice is negative
print("\n--- Investigating Negative UnitPrices ---")
negative_price_df = df[df['UnitPrice'] < 0]
print(negative_price_df[['InvoiceNo', 'Description', 'Quantity', 'UnitPrice', 'Missing_CustomerID']])


--- Negative Quantities: Missing CustomerID vs. Cancellations ---
Starts with "C" (True/False)     False  True 
Missing CustomerID (True/False)              
False                                0   8905
True                              1336    383

--- Investigating Negative UnitPrices ---
       InvoiceNo      Description  Quantity  UnitPrice  Missing_CustomerID
299983   A563186  Adjust bad debt         1  -11062.06                True
299984   A563187  Adjust bad debt         1  -11062.06                True


Customer ID is missingness is MAR. If we look at missing IDs descriptions and prices(as we did in the 2 code blocks above), we will see the connection. when missing IDs are excluded, min is negative price which is a anomaly, and in some missing IDs, there are system created descriptions like "bad debt adjustment" and so on. So, customer IDs missingness are related to these columns

In [8]:
# 1. Isolate only the rows where Description is missing
missing_desc_df = df[df['Description'].isnull()]

print("\n--- 1. What is the UnitPrice when Description is missing? ---")
# Let's look at the basic stats of the price for these specific rows
print(missing_desc_df['UnitPrice'].describe())

print("\n--- 2. Is there a CustomerID when Description is missing? ---")
# Count how many of these rows are ALSO missing a CustomerID
missing_cust_count = missing_desc_df['CustomerID'].isnull().sum()
total_missing_desc = len(missing_desc_df)
print(f"Out of {total_missing_desc} missing descriptions, {missing_cust_count} are also missing a CustomerID.")

print("\n--- 3. Let's look at the raw data ---")
# Display the first 5 rows to see what the Quantity and StockCode look like
print(missing_desc_df[['StockCode', 'Quantity', 'UnitPrice', 'CustomerID']].head())


--- 1. What is the UnitPrice when Description is missing? ---
count    1454.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: UnitPrice, dtype: float64

--- 2. Is there a CustomerID when Description is missing? ---
Out of 1454 missing descriptions, 1454 are also missing a CustomerID.

--- 3. Let's look at the raw data ---
     StockCode  Quantity  UnitPrice  CustomerID
622      22139        56        0.0         NaN
1970     21134         1        0.0         NaN
1971     22145         1        0.0         NaN
1972     37509         1        0.0         NaN
1987    85226A         1        0.0         NaN


missing descriptions is MAR because every missing description's unit price is zero and has empty customer ID, so we can predict description missingness by looking at unitprice and ID most of the time.  

missing value strategy:

for missing customer IDs, I chosed to drop these rows because there is no reasonable and senseful way to fill them for good.

for missing descriptions, I checked that missing descriptioned product Stockcode occurs somewhere else in database and there is a description for the product, so we can fill missing descriptions by using these stock codes

In [9]:
# Assuming your dataframe is loaded as 'df'
print(f"Original dataset shape: {df.shape}")
print(f"Missing Descriptions before: {df['Description'].isnull().sum()}")
print(f"Missing CustomerIDs before: {df['CustomerID'].isnull().sum()}")

# ==========================================
# STEP 1: Impute missing Descriptions
# ==========================================

# A. Create a "dictionary" (lookup table) of correct descriptions
# We take rows with valid descriptions, keep one unique description per StockCode, and convert it to a dictionary.
stock_desc_mapping = df.dropna(subset=['Description']).drop_duplicates(subset=['StockCode']).set_index('StockCode')['Description'].to_dict()

# B. Fill the missing values
# We map the missing descriptions to their corresponding StockCode in our dictionary
df['Description'] = df['Description'].fillna(df['StockCode'].map(stock_desc_mapping))

print(f"\nMissing Descriptions after imputation: {df['Description'].isnull().sum()}")

# ==========================================
# STEP 2: Drop missing CustomerIDs
# ==========================================

# Drop any row where the CustomerID column has a NaN (missing value)
df = df.dropna(subset=['CustomerID'])

# ==========================================
# STEP 3: Verify the Final Clean Dataset
# ==========================================
print("\n--- Final Cleaned Dataset ---")
print(f"Final dataset shape: {df.shape}")
print(f"Remaining missing values:\n{df.isnull().sum()}")

Original dataset shape: (541909, 12)
Missing Descriptions before: 1454
Missing CustomerIDs before: 135080

Missing Descriptions after imputation: 112

--- Final Cleaned Dataset ---
Final dataset shape: (406829, 12)
Remaining missing values:
InvoiceNo              0
StockCode              0
Description            0
Quantity               0
InvoiceDate            0
UnitPrice              0
CustomerID             0
Country                0
Missing_CustomerID     0
Missing_Description    0
InvoiceNo_Str          0
Is_Cancelled           0
dtype: int64


In [10]:
# 1. Convert InvoiceNo to a string (just in case pandas read some as pure numbers)
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

# 2. Filter rows where InvoiceNo starts with 'C'
cancellations_df = df[df['InvoiceNo'].str.startswith('C')]

# 3. Count them
cancel_count = len(cancellations_df)
total_rows = len(df)
cancel_pct = (cancel_count / total_rows) * 100

print(f"Number of cancellation transactions: {cancel_count:,}")
print(f"Percentage of total dataset: {cancel_pct:.2f}%")

Number of cancellation transactions: 8,905
Percentage of total dataset: 2.19%


We should flag cancellation rows by creating a dedicated Is_Cancelled boolean column. Removing them entirely would destroy valuable behavioral signals necessary for churn prediction (as high return rates indicate dissatisfaction). However, they must be flagged so they can be filtered out or mathematically netted against original purchases when calculating total revenue or training a recommender system.

In [11]:
# Make sure we have our Is_Cancelled flag
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['Is_Cancelled'] = df['InvoiceNo'].str.startswith('C')

# --- 1. Quantity <= 0 (Not Cancellations) ---
weird_qty_df = df[(df['Quantity'] <= 0) & (~df['Is_Cancelled'])]
weird_qty_count = len(weird_qty_df)

print(f"1. Quantity <= 0 (Not 'C'): {weird_qty_count} rows")
if weird_qty_count > 0:
    print(weird_qty_df[['InvoiceNo', 'StockCode', 'Description', 'Quantity']].head(3))

# --- 2. UnitPrice <= 0 ---
free_or_negative_price = df[df['UnitPrice'] <= 0]
price_issue_count = len(free_or_negative_price)

print(f"\n2. UnitPrice <= 0: {price_issue_count} rows")
if price_issue_count > 0:
    print(free_or_negative_price[['InvoiceNo', 'Description', 'UnitPrice']].head(3))

# --- 3. Extreme Outliers (Using the 99.9th Percentile as a threshold) ---
# We use 99.9% to find the truly ridiculous values, not just slightly large orders
qty_threshold = df['Quantity'].quantile(0.999)
price_threshold = df['UnitPrice'].quantile(0.999)

extreme_qty_count = len(df[df['Quantity'] > qty_threshold])
extreme_price_count = len(df[df['UnitPrice'] > price_threshold])

print(f"\n3. Extreme Outliers:")
print(f"   Quantity > {qty_threshold:.2f}: {extreme_qty_count} rows")
print(f"   UnitPrice > £{price_threshold:.2f}: {extreme_price_count} rows")

1. Quantity <= 0 (Not 'C'): 0 rows

2. UnitPrice <= 0: 40 rows
      InvoiceNo                   Description  UnitPrice
9302     537197  ROUND CAKE TIN VINTAGE GREEN        0.0
33576    539263  ADVENT CALENDAR GINGHAM SACK        0.0
40089    539722      REGENCY CAKESTAND 3 TIER        0.0

3. Extreme Outliers:
   Quantity > 504.00: 393 rows
   UnitPrice > £50.00: 388 rows


Records with negative or zero quantity that are not officially flagged as cancellations represent data corruption or system artifacts. Retaining them will artificially lower total sales volume calculations. Therefore, these rows should be removed entirely.

Records with a UnitPrice 0.00 or less distort financial metrics and customer lifetime value. Free items do not reflect a customer's true willingness to pay, and negative prices are ledger adjustments. These rows should be removed to ensure accurate revenue modeling.

If a customer buys 80,000 of a single item, keeping that massive number will completely mess up our averages for normal everyday shoppers. Instead of deleting these big spenders, we cap their quantity at a reasonable maximum limit. On the other hand, extremely high prices (like £8,000) in this dataset are usually just administrative fees or postage costs, not actual physical products. We should remove those fee rows so our analysis focuses purely on real store items.

In [12]:
# Print shape before we start this final cleaning phase
shape_before = df.shape
print(f"Shape before cleaning: {shape_before}")

# 1. Make sure our cancellation flag is set
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['Is_Cancelled'] = df['InvoiceNo'].str.startswith('C')

# 2. Remove Quantity <= 0 (BUT keep the official cancellations)
# This keeps rows where Quantity is positive OR it is a flagged cancellation
df = df[(df['Quantity'] > 0) | (df['Is_Cancelled'])]

# 3. Remove zero or negative UnitPrices
df = df[df['UnitPrice'] > 0]

# 4. Handle Extreme Outliers
# Find the limits (using 99.9th percentile for price, 99th for quantity)
price_upper_limit = df['UnitPrice'].quantile(0.999)
qty_upper_limit = df['Quantity'].quantile(0.99)

# Remove the extreme prices (fees, postage, etc.)
df = df[df['UnitPrice'] <= price_upper_limit]

# Cap the extreme quantities at the 99th percentile limit
# We use .loc to safely update only the rows that exceed the limit
df.loc[df['Quantity'] > qty_upper_limit, 'Quantity'] = qty_upper_limit

# Print shape after cleaning
shape_after = df.shape
print(f"Shape after cleaning: {shape_after}")
print(f"Total rows removed in this step: {shape_before[0] - shape_after[0]:,}")

# ==========================================
# VERIFICATION CHECKS
# ==========================================
print("\n--- Verification ---")

# Verify 1: No remaining negative/zero quantities (unless they are cancellations)
invalid_qty_count = len(df[(df['Quantity'] <= 0) & (~df['Is_Cancelled'])])
print(f"Negative/zero quantities (not cancelled): {invalid_qty_count}")

# Verify 2: No zero or negative prices
invalid_price_count = len(df[df['UnitPrice'] <= 0])
print(f"Zero/negative prices: {invalid_price_count}")

Shape before cleaning: (406829, 12)
Shape after cleaning: (406401, 12)
Total rows removed in this step: 428

--- Verification ---
Negative/zero quantities (not cancelled): 0
Zero/negative prices: 0


In [15]:
unique_countries = df['Country'].nunique()
print(f"Unique countries: {unique_countries}")

# Calculate percentage of transactions from Top 5 countries
top_5_pct = df['Country'].value_counts(normalize=True).head(5).sum() * 100
print(f"Percentage of transactions from Top 5 countries: {top_5_pct:.2f}%")

# Find countries with fewer than 50 transactions
country_counts = df['Country'].value_counts()
rare_countries = country_counts[country_counts < 50].index
print(f"Number of countries with < 50 transactions: {len(rare_countries)}")

# Create cleaned Country column
df['Country_Cleaned'] = df['Country'].apply(lambda x: 'Other' if x in rare_countries else x)

print(f"Categories BEFORE grouping: {unique_countries}")
print(f"Categories AFTER grouping: {df['Country_Cleaned'].nunique()}")


# ==========================================
# 3.2 — Analyze the StockCode column
# ==========================================
print("\n--- 3.2 StockCode Analysis ---")

# Convert to string just in case
df['StockCode_Str'] = df['StockCode'].astype(str)

unique_stock_codes = df['StockCode_Str'].nunique()
print(f"Unique stock codes: {unique_stock_codes}")

# Normal products in this dataset are usually 5-6 digit numbers (sometimes with a letter at the end).
# Non-product codes are often purely alphabetical (e.g., 'POST', 'M', 'D', 'DOT').
non_product_mask = df['StockCode_Str'].str.contains(r'^[a-zA-Z\s]+$')
non_product_codes = df[non_product_mask]['StockCode_Str'].unique()

print(f"Identified non-product codes: {list(non_product_codes)}")


# ==========================================
# 3.3 — Engineer a feature from Description
# ==========================================
print("\n--- 3.3 Description Feature Engineering ---")

# Ensure all descriptions are strings
df['Description'] = df['Description'].astype(str)

# Create a boolean flag if the item is a "SET" or "PACK"
df['Is_Bundle'] = df['Description'].str.contains('SET|PACK|PAIR', case=False, na=False)

# Show the relationship between this new feature and Quantity/UnitPrice
bundle_stats = df.groupby('Is_Bundle')[['Quantity', 'UnitPrice']].mean()
print("\nRelationship between Bundle status and metrics (Mean):")
print(bundle_stats.round(2))

Unique countries: 37
Percentage of transactions from Top 5 countries: 95.85%
Number of countries with < 50 transactions: 6
Categories BEFORE grouping: 37
Categories AFTER grouping: 32

--- 3.2 StockCode Analysis ---
Unique stock codes: 3678
Identified non-product codes: ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT', 'CRUK']

--- 3.3 Description Feature Engineering ---

Relationship between Bundle status and metrics (Mean):
           Quantity  UnitPrice
Is_Bundle                     
False          9.87       2.96
True          10.81       2.62


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Ensure InvoiceDate is a datetime object
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# ==========================================
# 4.1 — Engineer a binary target
# ==========================================
print("--- 4.1 Customer Aggregation ---")

# Calculate revenue per row
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Aggregate to customer level
customer_df = df.groupby('CustomerID').agg(
    total_revenue=('Revenue', 'sum'),
    num_orders=('InvoiceNo', 'nunique'),
    num_products=('StockCode', 'nunique'),
    first_purchase=('InvoiceDate', 'min'),
    last_purchase=('InvoiceDate', 'max')
).reset_index()

# Calculate customer tenure in days (behavioral feature)
customer_df['tenure_days'] = (customer_df['last_purchase'] - customer_df['first_purchase']).dt.days

# Define High-Value Threshold (Top 10%)
threshold = customer_df['total_revenue'].quantile(0.90)
customer_df['high_value'] = (customer_df['total_revenue'] >= threshold).astype(int)

print(f"Revenue threshold for Top 10%: £{threshold:.2f}")
print(f"Customer dataset shape: {customer_df.shape}")


# ==========================================
# 4.2 — Measure the imbalance
# ==========================================
print("\n--- 4.2 Class Imbalance ---")

class_dist = customer_df['high_value'].value_counts(normalize=True) * 100
print(f"Class Distribution:\n{class_dist.round(2)}")

baseline_accuracy = class_dist[0]
print(f"Baseline Accuracy (predicting 'regular' every time): {baseline_accuracy:.2f}%")


# ==========================================
# 4.3 — Apply resampling
# ==========================================
print("\n--- 4.3 Resampling & Modeling ---")

# Define Features (X) and Target (y). Note: Exclude total_revenue to prevent data leakage!
X = customer_df[['num_orders', 'num_products', 'tenure_days']]
y = customer_df['high_value']

# Split the data (stratify ensures the 90/10 split is maintained in both train and test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Original Training Set Distribution: {np.bincount(y_train)}")

# 1. Random Oversampling
ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)
print(f"Oversampled Training Set Distribution: {np.bincount(y_train_over)}")

# 2. Random Undersampling
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)
print(f"Undersampled Training Set Distribution: {np.bincount(y_train_under)}")



# --- Train & Evaluate Models ---
def evaluate_model(X_tr, y_tr, name):
    model = LogisticRegression(random_state=42)
    model.fit(X_tr, y_tr)
    # Always evaluate on the original, untouched test set!
    y_pred = model.predict(X_test) 
    
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"\n{name} Performance (Class 1):")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall:    {recall:.3f}")
    print(f"  F1-Score:  {f1:.3f}")

evaluate_model(X_train, y_train, "Original (No Resampling)")
evaluate_model(X_train_over, y_train_over, "Oversampled")
evaluate_model(X_train_under, y_train_under, "Undersampled")

--- 4.1 Customer Aggregation ---
Revenue threshold for Top 10%: £3429.61
Customer dataset shape: (4360, 8)

--- 4.2 Class Imbalance ---
Class Distribution:
high_value
0    90.0
1    10.0
Name: proportion, dtype: float64
Baseline Accuracy (predicting 'regular' every time): 90.00%

--- 4.3 Resampling & Modeling ---
Original Training Set Distribution: [3139  349]
Oversampled Training Set Distribution: [3139 3139]
Undersampled Training Set Distribution: [349 349]

Original (No Resampling) Performance (Class 1):
  Precision: 0.831
  Recall:    0.563
  F1-Score:  0.671

Oversampled Performance (Class 1):
  Precision: 0.547
  Recall:    0.862
  F1-Score:  0.670

Undersampled Performance (Class 1):
  Precision: 0.556
  Recall:    0.851
  F1-Score:  0.673


We grouped any country with fewer than 50 transactions into a single "Other" category. High cardinality in categorical variables can cause models (like decision trees or regressions) to overfit on tiny sample sizes. By reducing the number of categories, we consolidate low-variance data, making the Country feature much more robust for machine learning without losing the "international vs. domestic" signal

we will remove these stockcodes. If the goal is a product-level analysis (like building a product recommendation engine or market basket analysis), administrative fees and postage are irrelevant noise. You cannot "recommend" a manual accounting adjustment to a customer alongside a coffee mug.

We scanned the free-text Description column for keywords like "SET", "PACK", or "PAIR" and flagged those rows as True.
When grouping the data by this new feature, you will see a meaningful difference in the averages. "Bundled" items typically have a different average UnitPrice and are bought in different average Quantities compared to single, standalone items. This proves the feature captures real-world purchasing behavior and will be highly valuable for a pricing or segmentation model.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Ensure Revenue exists and InvoiceDate is datetime
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# ==========================================
# 5.1 — The WRONG Way (Intentional Leakage)
# ==========================================
print("--- 5.1 & 5.2: The Leaked Model ---")

# 1. We want to predict if they buy in Oct-Dec 2011.
prediction_start = pd.Timestamp("2011-10-01")

# WRONG: We calculate features using the ENTIRE dataset (including the future!)
leak_df = df.groupby('CustomerID').agg(
    total_spend=('Revenue', 'sum'),
    num_orders=('InvoiceNo', 'nunique'),
    last_purchase_date=('InvoiceDate', 'max') # BIG MISTAKE: This looks into the future!
).reset_index()

# Convert the date into a numeric feature (Days since data collection started)
leak_df['last_purchase_days'] = (leak_df['last_purchase_date'] - pd.Timestamp("2010-12-01")).dt.days

# Define Target: Did they make any purchase on or after Oct 1st?
target_buyers = df[df['InvoiceDate'] >= prediction_start]['CustomerID'].unique()
leak_df['bought_in_target_window'] = leak_df['CustomerID'].isin(target_buyers).astype(int)

# Detect the Leakage (Correlation)
print("\nFeature Correlations with Target (Look for suspiciously high numbers!):")
print(leak_df[['total_spend', 'num_orders', 'last_purchase_days', 'bought_in_target_window']].corr()['bought_in_target_window'])

# Train the Leaked Model
X_leak = leak_df[['total_spend', 'num_orders', 'last_purchase_days']]
y_leak = leak_df['bought_in_target_window']
X_train_L, X_test_L, y_train_L, y_test_L = train_test_split(X_leak, y_leak, test_size=0.2, random_state=42)

model_leak = LogisticRegression()
model_leak.fit(X_train_L, y_train_L)
print(f"\nLeaked Model Accuracy:  {accuracy_score(y_test_L, model_leak.predict(X_test_L)):.3f}")
print(f"Leaked Model Precision: {precision_score(y_test_L, model_leak.predict(X_test_L)):.3f}")

# ==========================================
# 5.3 — The RIGHT Way (Temporal Split)
# ==========================================
print("\n--- 5.3: The Correct Temporal Model ---")


observation_end = pd.Timestamp("2011-09-30")

# Split the RAW data temporally
df_obs = df[df["InvoiceDate"] <= observation_end]
df_pred = df[df["InvoiceDate"] >= prediction_start]

# RIGHT: Calculate features ONLY using the observation window (Past)
clean_df = df_obs.groupby('CustomerID').agg(
    total_spend=('Revenue', 'sum'),
    num_orders=('InvoiceNo', 'nunique'),
    last_purchase_date=('InvoiceDate', 'max')
).reset_index()

clean_df['last_purchase_days'] = (clean_df['last_purchase_date'] - pd.Timestamp("2010-12-01")).dt.days

# Define Target using ONLY the prediction window (Future)
target_customers_clean = df_pred['CustomerID'].unique()
clean_df['bought_in_target_window'] = clean_df['CustomerID'].isin(target_customers_clean).astype(int)

# Train the Correct Model
X_clean = clean_df[['total_spend', 'num_orders', 'last_purchase_days']]
y_clean = clean_df['bought_in_target_window']
X_train_C, X_test_C, y_train_C, y_test_C = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

model_clean = LogisticRegression()
model_clean.fit(X_train_C, y_train_C)
print(f"Correct Model Accuracy:  {accuracy_score(y_test_C, model_clean.predict(X_test_C)):.3f}")
print(f"Correct Model Precision: {precision_score(y_test_C, model_clean.predict(X_test_C)):.3f}")

--- 5.1 & 5.2: The Leaked Model ---

Feature Correlations with Target (Look for suspiciously high numbers!):
total_spend                0.131653
num_orders                 0.245973
last_purchase_days         0.804842
bought_in_target_window    1.000000
Name: bought_in_target_window, dtype: float64

Leaked Model Accuracy:  1.000
Leaked Model Precision: 1.000

--- 5.3: The Correct Temporal Model ---
Correct Model Accuracy:  0.703
Correct Model Precision: 0.733


Leaked Model & Detection: The first model performed suspiciously well because it cheated. By building features using the entire dataset, features like last_purchase_date included data from October through December. The correlation check proved the leak: the model was literally looking at future purchase dates to predict if a customer bought something in the future.
Why the Temporal Split is Correct: In the real world, you only have past data to predict future actions. A temporal split fixes the leak by strictly dividing the timeline. We use only past behavior (Dec 2010 – Sep 2011) as our features to predict a future outcome (Oct 2011 – Dec 2011). This stops the model from peeking at the answers and forces it to actually predict, giving us realistic and trustworthy results.